# ML-04 — Data Contract & Feature Leakage
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

This notebook develops on March 2026, a mid-panel month. June 2026 is treated as the sealed final month.


## 1. Data Contract — five plain-language answers

1. **What one row means:** one row represents one observed content-page performance record for one report date, one pseudonymized client, and one pseudonymized content item.
2. **Which table(s):** `fact_content_daily_performance`.
3. **Time window:** March 2026 for development and verification; June 2026 remains sealed.
4. **What I will rank:** content pages for refresh-review priority using observed performance signals. This is decision support, not causal prediction.
5. **One deliberate exclusion:** future-window outcome fields and label-derived fields are excluded from the honest feature set.


In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError("Add a Hugging Face READ token as a Colab Secret named HF_TOKEN.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MARCH = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Development month: March 2026")


## 2. Three verification queries

Exactly three verification queries are used below.


### Query 1 — Grain


In [ ]:
q1 = con.sql(f'''
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT report_date || '|' || client_id || '|' || content_id) AS distinct_grain_keys,
    COUNT(*) = COUNT(DISTINCT report_date || '|' || client_id || '|' || content_id) AS grain_matches
FROM {MARCH}
''').df()
display(q1)


### Query 2 — March 2026 row count and date span


In [ ]:
q2 = con.sql(f'''
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date,
    COUNT(DISTINCT report_date) AS distinct_report_dates
FROM {MARCH}
''').df()
display(q2)


### Query 3 — Availability (`IS TRUE`)


In [ ]:
q3 = con.sql(f'''
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {MARCH}
''').df()
display(q3)


## 3. Five-feature frame

Five features maximum. Each is knowable at the decision moment because it comes from information already observed before the decision.


In [ ]:
schema = con.sql(f"DESCRIBE SELECT * FROM {MARCH}").df()
print(schema[["column_name", "column_type"]].to_string(index=False))


### Feature availability notes

- `impressions` — knowable at the decision moment because recorded search impressions are already observed.
- `clicks` — knowable at the decision moment because recorded clicks are already observed.
- `sessions` — knowable at the decision moment because recorded sessions are already observed.
- `engagement_rate` — knowable at the decision moment because it is derived from observed engagement/session data.
- `days_since_last_update` — knowable at the decision moment because it is derived from observed content-update history.


In [ ]:
cols = set(schema["column_name"].tolist())

def pick(*names):
    for name in names:
        if name in cols:
            return name
    raise KeyError("Missing expected warehouse column. Tried: " + ", ".join(names))

IMP = pick("gsc_impressions", "impressions")
CLK = pick("gsc_clicks", "clicks")
SES = pick("ga4_sessions", "sessions")
ENG = pick("engagement_rate")
AGE = pick("days_since_last_update", "content_age_days")

feature_sql = f'''
SELECT
    client_id,
    content_id,
    {IMP} AS impressions,
    {CLK} AS clicks,
    {SES} AS sessions,
    {ENG} AS engagement_rate,
    {AGE} AS days_since_last_update
FROM {MARCH}
WHERE gsc_data_available IS TRUE
LIMIT 10000
'''

feature_frame = con.sql(feature_sql).df()
print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())


## 4. Deliberate leakage trap

For the experiment only, create one future/label-derived column: the next-month change in impressions. It is unavailable at the March decision moment. The leaked field is then removed.


In [ ]:
march_agg = con.sql(f'''
SELECT client_id, content_id, SUM({IMP}) AS impressions_march
FROM {MARCH}
GROUP BY client_id, content_id
''').df()

april_agg = con.sql(f'''
SELECT client_id, content_id, SUM({IMP}) AS impressions_april
FROM {APRIL}
GROUP BY client_id, content_id
''').df()

trap = march_agg.merge(april_agg, on=["client_id", "content_id"], how="inner")
trap["future_impression_change"] = (
    trap["impressions_april"] - trap["impressions_march"]
)

print("Deliberate leakage column:", "future_impression_change")
display(trap.head())


In [ ]:
# Deliberately leaky quick score: it directly uses future outcome information.
trap["leaky_score"] = trap["future_impression_change"].abs()

print("Leaky score summary:")
print(trap["leaky_score"].describe())


In [ ]:
# Remove the deliberate leak and keep only the honest five-feature frame.
honest_feature_frame = feature_frame.copy()
assert "future_impression_change" not in honest_feature_frame.columns

print("Leakage removed.")
print("Honest feature columns:")
print(honest_feature_frame.columns.tolist())


## 5. Limitation

This lane is useful for decision support, but observed performance signals do not establish causality. The warehouse is an unbalanced panel, availability can differ across measurement systems, and a refresh-priority score does not prove that refreshing a page will improve future performance. Human review is required before action.


## 6. Self-check

- Five plain-language contract answers
- Exactly three verification queries with visible outputs
- Availability checked with `IS TRUE`
- Five features maximum
- "Knowable at the decision moment" explanation for each feature
- One deliberate label/future-derived leakage column
- Leakage removed from the honest feature set
- One named limitation
- Notebook executed and committed to `work/notebooks/w03_data_contract.ipynb`
